## Notebook Purpose

This notebook (**Notebook 2 of 4** in the research pipeline) contains standalone
utility/helper scripts used to clean, anonymize, reformat, and merge the raw dataset
files produced by the generation notebook (`arj-inception-2.ipynb`) before they are
used for benchmarking and fine-tuning.

Unlike the other notebooks, this one is not meant to be run top-to-bottom in a single
pass — each cell is a self-contained utility that was run as-needed during dataset
preparation (anonymizing names, reformatting schemas, merging contributor files,
de-duplicating records). Treat this notebook as a **toolbox**, and re-run only the
cell(s) relevant to the current data-cleaning task.

**Why this step is necessary:** the raw generated dataset (from Notebook 1) and
additional contributions from collaborators arrive in slightly different shapes /
naming conventions. Before the data can be used for benchmarking (Notebook 3) or
fine-tuning (Notebook 4), it must be normalized into one consistent schema and file.

**Note:** this first cell calls `process_pm_datasets(...)`, a function that is not
defined anywhere in this notebook (only `process_datasets`, defined in the next code
cell, is). This looks like a leftover/typo from an earlier version of the script —
run the next cell (which defines `process_datasets`) first, and call that function
name instead, or define `process_pm_datasets` yourself before running this cell.


In [ ]:
import os

# Check if there are any json/jsonl files in /content
input_path = '/content/'
files = [f for f in os.listdir(input_path) if f.endswith('.json') or f.endswith('.jsonl')]
print(f"Found files to process: {files}")

# Correcting the paths to use the Colab environment standard
process_pm_datasets(
    input_dir='/content/',
    team_pool=my_team_pool,
    output_dir='/content/processed_role_data'
)

Found files to process: ['pm_val.jsonl', 'pm_test.jsonl', 'pm_train.jsonl']
Processing: pm_val.jsonl
Processing: pm_test.jsonl
Processing: pm_train.jsonl

Success! Modified files saved to: /content/processed_role_data


### New code

### Name Anonymization Utility

Defines `create_alias_map()` and `process_datasets()`, used to strip real personal
names out of generated PM examples and replace them with role labels (e.g.
"Arjun Sharma" → "Senior Backend Engineer").

**Methodology:**
1. `create_alias_map(team_pool)` builds a lookup table from full names, and
   unambiguous first/last names, to each person's role. Names that are ambiguous
   (shared by more than one team member) are deliberately excluded to avoid
   incorrect replacements.
2. `process_datasets(input_dir, team_pool, output_dir)` reads every `.json`/`.jsonl`
   file in `input_dir`, replaces every alias occurrence (longest match first, using
   word-boundary regex so "Arjun" doesn't match inside "Arjunav") with the
   corresponding role string, and writes the anonymized file to `output_dir`.

**Input:** raw dataset files containing team-member names; a `team_pool` list of
`{name, role, skills, capacity}` dicts (a large example roster, `team_data`, is
defined below for reference/testing).

**Output:** the same files, with personal names replaced by role labels, written to
`output_dir`.

**Why this step is necessary:** replacing names with role labels keeps the dataset
free of personally identifiable information while preserving the PM-relevant
signal (who was assigned what type of role), which is what the downstream models
are meant to reason about.


In [ ]:
import json
import os
import glob
import re
from collections import Counter

def create_alias_map(team_pool):
    """
    Creates a mapping of Full Names, First Names, and Last Names to Roles.
    Skips first/last names if they are shared by multiple people to avoid errors.
    """
    full_name_map = {}
    first_name_list = []
    last_name_list = []

    # 1. Map full names and collect first/last names
    for member in team_pool:
        name = member['name']
        role = member['role']
        full_name_map[name] = role

        parts = name.split()
        if len(parts) >= 2:
            first_name_list.append((parts[0], role))
            last_name_list.append((parts[-1], role))

    # 2. Find ambiguous names (names shared by >1 person)
    first_counts = Counter([x[0] for x in first_name_list])
    last_counts = Counter([x[0] for x in last_name_list])

    # 3. Build the final replacement dictionary
    final_map = dict(full_name_map)

    # Add first names if they aren't ambiguous
    for first, role in first_name_list:
        if first_counts[first] == 1 and first not in final_map:
            final_map[first] = role

    # Add last names if they aren't ambiguous
    for last, role in last_name_list:
        if last_counts[last] == 1 and last not in final_map:
            final_map[last] = role

    return final_map

def process_datasets(input_dir, team_pool, output_dir):
    alias_map = create_alias_map(team_pool)

    # Sort keys by length descending to ensure "Arjun Sharma" is
    # replaced before "Arjun"
    sorted_keys = sorted(alias_map.keys(), key=len, reverse=True)

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    files = glob.glob(os.path.join(input_dir, "*.json*"))

    for file_path in files:
        filename = os.path.basename(file_path)
        output_path = os.path.join(output_dir, filename)

        with open(file_path, 'r', encoding='utf-8') as f_in, \
             open(output_path, 'w', encoding='utf-8') as f_out:

            # Use same logic for JSON and JSONL by processing the string content
            content = f_in.read()

            # Apply replacements using regex word boundaries (\b)
            # This ensures 'Arjun' doesn't match 'Arjunav' or 'Arjun123'
            for name_key in sorted_keys:
                role_val = alias_map[name_key]
                # Regex \b matches the position between a word char and a non-word char
                pattern = rf'\b{re.escape(name_key)}\b'
                content = re.sub(pattern, role_val, content)

            f_out.write(content)

    print(f"Transformation complete. Replaced {len(sorted_keys)} name variations.")

# --- EXAMPLE USAGE ---
team_data = [
        {"name": "Arjun Sharma",    "role": "Senior Backend Engineer",  "skills": ["Python", "Django", "PostgreSQL", "Redis"], "capacity": 8},
    {"name": "Priya Nair",      "role": "Frontend Engineer",        "skills": ["React", "TypeScript", "CSS", "GraphQL"],  "capacity": 8},
    {"name": "Mohammed Al-Razi","role": "DevOps Engineer",          "skills": ["Kubernetes", "Terraform", "CI/CD", "AWS"],"capacity": 8},
    {"name": "Sarah Chen",      "role": "ML Engineer",              "skills": ["PyTorch", "MLflow", "Python", "Spark"],   "capacity": 8},
    {"name": "Raj Patel",       "role": "QA Engineer",              "skills": ["Selenium", "Jest", "Cypress", "Postman"], "capacity": 6},
    {"name": "Lisa Nguyen",     "role": "UI/UX Designer",           "skills": ["Figma", "Prototyping", "User Research"],  "capacity": 8},
    {"name": "Omar Hassan",     "role": "Data Engineer",            "skills": ["Spark", "Airflow", "dbt", "Snowflake"],   "capacity": 8},
    {"name": "Ananya Reddy",    "role": "Security Engineer",        "skills": ["OWASP", "Penetration Testing", "OAuth2"], "capacity": 6},
    {"name": "Carlos Mendes",   "role": "Mobile Engineer",          "skills": ["React Native", "Swift", "Kotlin"],        "capacity": 8},
    {"name": "Fatima Malik",    "role": "Junior Backend Engineer",  "skills": ["Python", "Flask", "MySQL"],               "capacity": 8},

    {"name": "Rohan Mehta",       "role": "Senior Backend Engineer",
     "skills": ["Go", "PostgreSQL", "Redis", "Microservices"], "capacity": 8},

    {"name": "Aisha Khan",        "role": "Frontend Engineer",
     "skills": ["React", "Next.js", "TypeScript", "Tailwind"], "capacity": 8},

    {"name": "Daniel Park",       "role": "DevOps Engineer",
     "skills": ["Kubernetes", "Helm", "Terraform", "GCP"], "capacity": 8},

    {"name": "Elena Petrova",     "role": "ML Engineer",
     "skills": ["TensorFlow", "Python", "MLflow", "Feature Engineering"], "capacity": 8},

    {"name": "Marcus Johnson",    "role": "QA Engineer",
     "skills": ["Playwright", "Cypress", "API Testing", "Test Automation"], "capacity": 6},

    {"name": "Sofia Martinez",    "role": "UI/UX Designer",
     "skills": ["Figma", "Design Systems", "User Research", "Wireframing"], "capacity": 8},

    {"name": "Wei Zhang",         "role": "Data Engineer",
     "skills": ["Spark", "Kafka", "Airflow", "BigQuery"], "capacity": 8},

    {"name": "Yusuf Al-Farsi",    "role": "Security Engineer",
     "skills": ["OAuth2", "Zero Trust", "Threat Modeling", "SAST"], "capacity": 6},

    {"name": "Lucas Silva",       "role": "Mobile Engineer",
     "skills": ["Flutter", "Dart", "Android", "iOS"], "capacity": 8},

    {"name": "Nina Ivanova",      "role": "Blockchain Engineer",
     "skills": ["Solidity", "Ethereum", "Smart Contracts", "Web3.js"], "capacity": 7},

    {"name": "Ethan Walker",      "role": "Platform Engineer",
     "skills": ["Docker", "Kubernetes", "CI/CD", "Observability"], "capacity": 8},

    {"name": "Priyanka Das",      "role": "Junior Backend Engineer",
     "skills": ["Python", "FastAPI", "MySQL", "REST APIs"], "capacity": 8},
]

# process_datasets('./input_folder', team_data, './output_folder')

### Apply Anonymization to a Specific Contributor's Data

A second, targeted invocation of the same anonymization pipeline — this run points at
one collaborator's already partially processed directory
(`processed_role_data/Pankaj`) and writes the fully anonymized output to `./Pankaj`.
Re-running this cell pattern (with different `input_dir`/`output_dir`) is how each
contributor's raw data batch was anonymized individually.


In [ ]:
import os

# Check if there are any json/jsonl files in /content
input_path = '/content/'
files = [f for f in os.listdir(input_path) if f.endswith('.json') or f.endswith('.jsonl')]
print(f"Found files to process: {files}")

# Correcting the paths to use the Colab environment standard
process_pm_datasets(
    input_dir='/content/processed_role_data/Pankaj',
    team_pool=my_team_pool,
    output_dir='/content/Pankaj'
)

Found files to process: ['2_pm_sprint_planning_topup.jsonl', 'anscombe.json', '1_pm_sprint_planning_topup.jsonl', '2_pm_resource_planning_topup.jsonl', '1_pm_resource_planning_topup.jsonl', 'pm_finetune_dataset.jsonl']
Processing: pm_val.jsonl
Processing: pm_test.jsonl
Processing: pm_train.jsonl
Processing: pm_finetune_dataset.jsonl

Success! Modified files saved to: /content/Pankaj


### Reformat Data for SLM Benchmarking

The dataset generated in Notebook 1 is stored in **Qwen2 chat format**
(`messages: [{role, content}, ...]`), which is convenient for fine-tuning but not
what the benchmarking notebook expects. This section converts examples into a flat
`{id, task_type, input, reference_output}` format used as the ground-truth reference
during automatic-metric and LLM-judge scoring.


### Data formating for evalutating SLM

### Flatten Chat-Format Examples to Benchmark Format

Reads `pm_finetune_dataset.jsonl` (Qwen2 chat-format examples), extracts the user
prompt and assistant response from each example's `messages` array, and rewrites them
into the flat `{id, task_type, input, reference_output}` schema expected by the
benchmarking notebook.

**Input:** `pm_finetune_dataset.jsonl` (chat-format).
**Output:** `pm_benchmark_dataset_aadi_incep.jsonl` (flat format).


In [ ]:
import json

input_file = "pm_finetune_dataset.jsonl"
output_file = "pm_benchmark_dataset_aadi_incep.jsonl"

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        data = json.loads(line)

        # Extract user and assistant content from the messages array
        user_msg = next(m["content"] for m in data["messages"] if m["role"] == "user")
        assistant_msg = next(m["content"] for m in data["messages"] if m["role"] == "assistant")

        # Rebuild into the notebook's expected flat format
        formatted_data = {
            "id": data["id"],
            "task_type": data["task_type"],
            "input": user_msg,
            "reference_output": assistant_msg
        }

        outfile.write(json.dumps(formatted_data) + '\n')

print("Data reformatted successfully!")

Data reformatted successfully!


### Merge Multiple Contributors' Dataset Files

Several team members independently generated additional examples (topping up
underrepresented task types, or contributing via a different generator model such as
Gemma). This cell concatenates all of those individual `.jsonl` contribution files
into one combined dataset file, skipping (with a warning) any file that isn't found
locally.

**Input:** a list of per-contributor `.jsonl` files (`files_to_merge`).
**Output:** `pm_finetune_merged_final.jsonl` — the combined raw dataset before
de-duplication.


In [ ]:
import json

files_to_merge = ['arj_pm_finetune_dataset.jsonl',
 '2_pm_sprint_planning_topup.jsonl',
 'pank_pm_finetune_dataset.jsonl',
 'gemma_1_pm_task_design.jsonl',
 '1_pm_sprint_planning_topup.jsonl',
 'gemma_1_pm_task_allocation.jsonl',
 'arj2_pm_finetune_dataset.jsonl',
 '2_pm_resource_planning_topup.jsonl',
 '1_pm_resource_planning_topup.jsonl',
 'gemma_2_pm_task_design.jsonl',
 'gemma_2_pm_task_allocation.jsonl',
 'aadi_pm_finetune_dataset.jsonl',]




output_merged_file = "pm_finetune_merged_final.jsonl"

merged_count = 0
with open(output_merged_file, 'w', encoding='utf-8') as outfile:
    for file_name in files_to_merge:
        try:
            with open(file_name, 'r', encoding='utf-8') as infile:
                for line in infile:
                    if line.strip():
                        outfile.write(line)
                        merged_count += 1
        except FileNotFoundError:
            print(f"Warning: {file_name} not found, skipping.")

print(f"Successfully merged {merged_count} records into {output_merged_file}")

Successfully merged 7844 records into pm_finetune_merged_final.jsonl


### List Files in Working Directory

A quick utility cell to inspect what files are currently present in the Colab/local
working directory — useful for confirming that expected contributor files exist
before running the merge step above, or that merge/dedup outputs were written
correctly.


In [ ]:
import os
ls = [os.listdir()][0]
ls

['.config',
 'arj_pm_finetune_dataset.jsonl',
 '2_pm_sprint_planning_topup.jsonl',
 'pank_pm_finetune_dataset.jsonl',
 'gemma_1_pm_task_design.jsonl',
 '.ipynb_checkpoints',
 '1_pm_sprint_planning_topup.jsonl',
 'gemma_1_pm_task_allocation.jsonl',
 'arj2_pm_finetune_dataset.jsonl',
 '2_pm_resource_planning_topup.jsonl',
 '1_pm_resource_planning_topup.jsonl',
 'gemma_2_pm_task_design.jsonl',
 'gemma_2_pm_task_allocation.jsonl',
 'aadi_pm_finetune_dataset.jsonl',
 'sample_data']

### De-duplicate Merged Dataset

Because multiple contributors' files were merged together, some records may be exact
duplicates (same `id`). This cell reads the merged file, keeps only the first
occurrence of each unique `id`, and overwrites the file in place with the
de-duplicated set.

**Input / Output:** `pm_finetune_merged_final.jsonl` (read and overwritten).

**Why this step is necessary:** this de-duplicated, merged file is the final dataset
artifact that feeds into both the fine-tuning notebook (as the source for the
train/val/test split) and, after benchmarking-format conversion, the SLM
benchmarking notebook — so duplicate records here would silently bias both
downstream experiments.


In [ ]:
import json

file_path = 'pm_finetune_merged_final.jsonl'
unique_data = []
seen_ids = set()

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                sample_id = data.get('id')
                if sample_id not in seen_ids:
                    unique_data.append(data)
                    seen_ids.add(sample_id)

    # Write the unique records back to the file
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in unique_data:
            f.write(json.dumps(item) + '\n')

    print(f"Successfully removed duplicates. Total unique samples remaining: {len(unique_data)}")

except FileNotFoundError:
    print(f"Error: {file_path} not found.")

Successfully removed duplicates. Total unique samples remaining: 5042
